In [1]:
import sqlite3, os, time
from pathlib import Path

In [ ]:
from app.db.errors import DBErrorKind
from app.db.sqlite_connection import unit_of_work
from app.services.naming import string_normalize
from app.db.sqlite_errors import translate_sqlite_error

In [3]:
DB_PATH = "test.db"

In [ ]:

# if os.path.exists(DB_PATH):
#     os.remove(DB_PATH)

In [13]:
def attempt(conn, label, sql, params):
    start = time.perf_counter()
    try:
        conn.execute(sql, params)
        print(f"{label:<28} Statement Okay after {time.perf_counter() - start: .2f}s")
        return True 
    except sqlite3.Error as err:
        print(f"{label:<28} Failed with {type(err).__name__} after {time.perf_counter() - start: .2f}s")
        print("| errorname: ", getattr(err, "sqlite_errorname", None),
              "\n| errorcode: ", getattr(err, "sqlite_errorcode", None),
              "\n| errormessage: ", err)
        return False 

def rows(conn):
    result = conn.execute("SELECT * FROM clients ORDER BY id").fetchall()
    return result

def show(conn):
    clients = rows(conn)
    notes = conn.execute("SELECT id, client_id, note FROM client_note ORDER BY id").fetchall()
    print(f"clients: {clients}\nnotes: {notes}\n")

In [ ]:
client_table_create = """
    CREATE TABLE clients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        name_standardized TEXT NOT NULL,
        modified_by TEXT NOT NULL,
        CONSTRAINT uq_client_name UNIQUE (name_standardized)
    )
"""

conn = sqlite3.connect(DB_PATH, autocommit=False)
conn.execute(client_table_create)

name = "Farm A"
modified_by = "Alex"

conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", (name.strip(), string_normalize(name), modified_by))
conn.commit()
conn.close()

In [49]:
conn = sqlite3.connect(DB_PATH, autocommit=False, timeout=1)
rows(conn)

[(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex')]

In [58]:
name = "FARM   a"
attempt(conn, "inserting 'FARM a'", "INSERT INTO clients (name, modified_by) VALUES (?, ?)", (name, "Test User"))

inserting 'FARM a'           Failed with IntegrityError after  0.00s
| errorname:  SQLITE_CONSTRAINT_NOTNULL 
| errorcode:  1299 
| errormessage:  NOT NULL constraint failed: clients.name_standardized


False

In [59]:
name = "Farm a"
attempt(conn, "updating 'FARM B' to 'FARM A'", "UPDATE clients SET name=?, name_standardized=?, modified_by=? WHERE id=?", (name, string_normalize(name) , "Test User", 3))

updating 'FARM B' to 'FARM A' Failed with IntegrityError after  0.00s
| errorname:  SQLITE_CONSTRAINT_UNIQUE 
| errorcode:  2067 
| errormessage:  UNIQUE constraint failed: clients.name_standardized


False

In [56]:
# conn.commit()
rows(conn)

[(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex')]

In [60]:
conn.close()

## connection management

In [24]:
with unit_of_work(DB_PATH) as conn:
    conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm     C", string_normalize("Farm     C"), "Test User 3"))
    

In [26]:
with unit_of_work(DB_PATH) as conn:
    print(rows(conn))

[(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex'), (4, 'Farm     C', 'farm_c', 'Test User 3')]


In [40]:
try:
    with unit_of_work(DB_PATH) as conn:
        conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm E", string_normalize("Farm E"), "Test User 5"))
        conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm E", string_normalize("Farm E"), "Test User 5"))
except BaseException as err:
    print("\n| error type:", type(err).__name__, 
            "\n| error name", getattr(err, "sqlite_errorname", None), 
            "\n| error code", getattr(err, "sqlite_errorcode", None),
            "\n| error message", err)
    


| error type: IntegrityError 
| error name SQLITE_CONSTRAINT_UNIQUE 
| error code 2067 
| error message UNIQUE constraint failed: clients.name_standardized


In [7]:
with unit_of_work(DB_PATH) as conn:
    print(rows(conn))

[(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex'), (4, 'Farm     C', 'farm_c', 'Test User 3'), (5, 'Farm D', 'farm_d', 'Test User 4')]


## error translation

In [9]:
try:
    with unit_of_work(DB_PATH) as conn:
        conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm E", string_normalize("Farm E"), "Test User 5"))
        conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm E", string_normalize("Farm E"), "Test User 5"))
except BaseException as err:
    print("\n| error type:", type(err).__name__, 
            "\n| error name", getattr(err, "sqlite_errorname", None), 
            "\n| error code", getattr(err, "sqlite_errorcode", None),
            "\n| error message", err)
    print("\n classification: ", translate_sqlite_error(err))


| error type: IntegrityError 
| error name SQLITE_CONSTRAINT_UNIQUE 
| error code 2067 
| error message UNIQUE constraint failed: clients.name_standardized

 classification:  unique


In [10]:
try:
    with unit_of_work(DB_PATH) as conn:
        conn.execute("INSERT INTO clients (name, modified_by) VALUES (?, ?)", ("Farm E", "Test User 5"))
except BaseException as err:
    print("\n| error type:", type(err).__name__, 
            "\n| error name", getattr(err, "sqlite_errorname", None), 
            "\n| error code", getattr(err, "sqlite_errorcode", None),
            "\n| error message", err)
    print("\n classification: ", translate_sqlite_error(err))


| error type: IntegrityError 
| error name SQLITE_CONSTRAINT_NOTNULL 
| error code 1299 
| error message NOT NULL constraint failed: clients.name_standardized

 classification:  not_null


In [11]:
try:
    with unit_of_work(DB_PATH) as conn:
        conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm E", string_normalize("Farm E"), "Test User 5"))
        raise ValueError("oops")
        conn.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm E", string_normalize("Farm E"), "Test User 5"))
except BaseException as err:
    print("\n| error type:", type(err).__name__, 
            "\n| error name", getattr(err, "sqlite_errorname", None), 
            "\n| error code", getattr(err, "sqlite_errorcode", None),
            "\n| error message", err)
    print("\n classification: ", translate_sqlite_error(err))


| error type: ValueError 
| error name None 
| error code None 
| error message oops

 classification:  unknown


In [12]:
fake_note_table_sql = """
    CREATE TABLE client_note (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        client_id   INTEGER NOT NULL,
        note        TEXT NOT NULL,
        CONSTRAINT fk_client_notes_client
            FOREIGN KEY (client_id) REFERENCES clients(id) ON DELETE RESTRICT
    )
"""
try: 
    with unit_of_work(DB_PATH) as conn:
        conn.execute(fake_note_table_sql)
        conn.execute("INSERT INTO client_note (client_id, note) VALUES (?, ?)", (2, "Testing delete"))
        print(rows(conn))
except BaseException as err:
    print("\n| error type:", type(err).__name__, 
            "\n| error name", getattr(err, "sqlite_errorname", None), 
            "\n| error code", getattr(err, "sqlite_errorcode", None),
            "\n| error message", err)
    print("\n classification: ", translate_sqlite_error(err))

[(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex'), (4, 'Farm     C', 'farm_c', 'Test User 3'), (5, 'Farm D', 'farm_d', 'Test User 4')]


In [14]:
with unit_of_work(DB_PATH) as conn:
    show(conn)

clients: [(2, 'Farm A', 'farm_a', 'Alex'), (3, 'Farm B', 'farm_b', 'Alex'), (4, 'Farm     C', 'farm_c', 'Test User 3'), (5, 'Farm D', 'farm_d', 'Test User 4')]
notes: [(1, 2, 'Testing delete')]



In [15]:
try:
    with unit_of_work(DB_PATH) as conn:
        conn.execute("DELETE FROM clients WHERE id=?", (2,))
except BaseException as err:
    print("\n| error type:", type(err).__name__, 
            "\n| error name", getattr(err, "sqlite_errorname", None), 
            "\n| error code", getattr(err, "sqlite_errorcode", None),
            "\n| error message", err)
    print("\n classification: ", translate_sqlite_error(err))


| error type: IntegrityError 
| error name SQLITE_CONSTRAINT_TRIGGER 
| error code 1811 
| error message FOREIGN KEY constraint failed

 classification:  foreign_key


In [16]:
try:
    with unit_of_work(DB_PATH) as conn:
        conn.execute("INSERT INTO client_note (client_id, note) VALUES (?, ?)", (000, "Orphan note"))
except BaseException as err:
    print("\n| error type:", type(err).__name__, 
            "\n| error name", getattr(err, "sqlite_errorname", None), 
            "\n| error code", getattr(err, "sqlite_errorcode", None),
            "\n| error message", err)
    print("\n classification: ", translate_sqlite_error(err))


| error type: IntegrityError 
| error name SQLITE_CONSTRAINT_FOREIGNKEY 
| error code 787 
| error message FOREIGN KEY constraint failed

 classification:  foreign_key


In [26]:
try:
    with unit_of_work(DB_PATH) as conn1:
        with unit_of_work(DB_PATH) as conn2:
            conn1.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm E", string_normalize("Farm E"), "Test User 5"))
            conn2.execute("INSERT INTO clients (name, name_standardized, modified_by) VALUES (?, ?, ?)", ("Farm f", string_normalize("Farm f"), "Test User 5"))
except BaseException as err:
    print("\n| error type:", type(err).__name__, 
            "\n| error name", getattr(err, "sqlite_errorname", None), 
            "\n| error code", getattr(err, "sqlite_errorcode", None),
            "\n| error message", err)
    print("\n classification: ", translate_sqlite_error(err))


| error type: OperationalError 
| error name SQLITE_BUSY 
| error code 5 
| error message database is locked

 classification:  busy
